# Weekly Project 02 - Image features

## Robot Tracking

It is recommended that you finish the exercises from Monday, before starting the project.

For this project you are given a video of some mobile robots (Robots.mp4). The task is now to track only the robots that are moving. Try to use both sparse and dense optical flow and compare the results.

For sparse optical flow, draw the tracked keypoints onto each frame and try to show the frames fast enough, such that it looks like a video.

For dense optical flow, represent the movement in any way you see fitting. For example by making a new image with the colors of each pixel representing the movement.



In [1]:
import cv2
import numpy as np

In [2]:
video_path = "Robots.mp4"
cap = cv2.VideoCapture(video_path)

Sparce Optical Flow

In [3]:
ret, frame1 = cap.read()
if not ret:
    cap.release()
    raise FileNotFoundError(f"Could not open {video_path}")

gray_frame1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
feat1 = cv2.goodFeaturesToTrack(gray_frame1, maxCorners=100, qualityLevel=0.3, minDistance=7)

trail = np.zeros_like(frame1)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
for _ in range(frame_count - 1):
    ret, frame2 = cap.read()

    if not ret:
        cv2.destroyAllWindows()
        break

    gray_frame2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    if feat1 is None or len(feat1) == 0:
        feat1 = cv2.goodFeaturesToTrack(gray_frame1, maxCorners=100, qualityLevel=0.3, minDistance=7)

    next_feat, status, err = cv2.calcOpticalFlowPyrLK(
        gray_frame1,
        gray_frame2,
        feat1,
        None
    )

    if next_feat is None:
        break

    old_feat = feat1[status == 1]
    next_feat = next_feat[status == 1]

    for old_point, new_point in zip(old_feat, next_feat):
        x_old, y_old = old_point.astype(int)
        x_new, y_new = new_point.astype(int)

        cv2.circle(frame2, (x_new, y_new), 6, (0, 255, 0), -1)
        cv2.line(trail, (x_old, y_old), (x_new, y_new), (0, 0, 255), 3)

    output = cv2.add(frame2, trail)
    cv2.imshow("Sparse optical flow", output)

    if cv2.waitKey(20) & 0xFF == ord("q"):
        break

    feat1 = next_feat.reshape(-1, 1, 2)
    gray_frame1 = gray_frame2.copy()

cap.release()
cv2.destroyAllWindows()

Dense optical flow

In [ ]:
ret, frame1 = cap.read()
if not ret:
    cap.release()
    raise FileNotFoundError(f"Could not read the first frame from {video_path}")

previous_gray = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

sampling = 30
arrow_scale = 3
motion_threshold = 1.0

while True:
    ret, frame2 = cap.read()
    if not ret:
        break

    current_gray = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    flow = cv2.calcOpticalFlowFarneback(
        previous_gray,
        current_gray,
        None,
        0.5,
        3,
        15,
        3,
        5,
        1.5,
        0
    )

    mag, ang = cv2.cartToPolar(
        flow[:, :, 0],
        flow[:, :, 1]
    )

    height, width = current_gray.shape

    sam_mag = mag[::sampling, ::sampling]
    sam_ang = ang[::sampling, ::sampling]

    start_y, start_x = np.mgrid[
        0:height:sampling,
        0:width:sampling
    ]

    end_x = start_x + arrow_scale * sam_mag * np.cos(sam_ang)
    end_y = start_y + arrow_scale * sam_mag * np.sin(sam_ang)

    end_x = np.clip(end_x, 0, width - 1).astype(int)
    end_y = np.clip(end_y, 0, height - 1).astype(int)

    # Draw the flow vectors on the current video frame
    output = frame2.copy()

    for i in range(sam_mag.shape[0]):
        for j in range(sam_mag.shape[1]):
            if sam_mag[i, j] > motion_threshold:
                start_point = (
                    int(start_x[i, j]),
                    int(start_y[i, j])
                )

                end_point = (
                    int(end_x[i, j]),
                    int(end_y[i, j])
                )

                cv2.arrowedLine(
                    output,
                    start_point,
                    end_point,
                    (0, 255, 0),
                    1,
                    tipLength=0.3
                )

    cv2.imshow("Dense optical flow", output)

    if cv2.waitKey(20) & 0xFF == ord("q"):
        break

    previous_gray = current_gray

cap.release()
cv2.destroyAllWindows()

: 